# Demo: Branching Skeleton

In [1]:
from mascaf import *

import logging
logging.basicConfig(level=logging.INFO)

In [2]:
mm = MeshManager(mesh_path="../../data/demo/test_branching_3.obj")
mm.print_mesh_analysis()
raw_skeleton = SkeletonGraph.from_txt(f"../../data/demo/test_branching_3.polylines.txt")
# raw_skeleton.prune_short_branches_inplace(min_length_percentile=1)
mm.visualize_mesh_3d(title="Branching test model", skel=raw_skeleton)

INFO:mascaf.mesh:Loaded mesh: 1417 vertices, 2830 faces
INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 1417
INFO:mascaf.mesh:  * Faces: 2830
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 5.11
INFO:mascaf.mesh:  * Bounds: [-1.0, -1.3, -3.6] to [1.0, 1.4, 1.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 0
INFO:mascaf.mesh:  * Euler Characteristic: 2
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


In [3]:
opts = SkeletonOptimizerOptions(
    max_iterations=5,
    step_size=0.01,
    smoothing_weight=0.1,
    preserve_terminal_nodes=False,
    preserve_branch_nodes=False,
    verbose=True,
)
print("\nOptimizing skeleton (serial)...")
optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
optimized_skeleton = optimizer.optimize()
f = mm.visualize_mesh_3d(
    skel=[raw_skeleton, optimized_skeleton],
    skel_color=["crimson", "blue"],
    show_axes=True,
)
f.show()

skeleton = optimized_skeleton

INFO:mascaf.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mascaf.skeleton_optimizer:Starting skeleton optimization...
INFO:mascaf.skeleton_optimizer:  Nodes: 137
INFO:mascaf.skeleton_optimizer:  Max iterations: 5
INFO:mascaf.skeleton_optimizer:  Step size: 0.0100
INFO:mascaf.skeleton_optimizer:  Smoothing weight: 0.1000



Optimizing skeleton (serial)...


INFO:mascaf.skeleton_optimizer:  Iteration 0: avg movement = 0.009481
INFO:mascaf.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mascaf.skeleton_optimizer:Optimization complete


In [4]:
max_edge_length = 0.5

swc_filepath = "../../data/demo/test_branching_3.swc"

radius_strategy = "equivalent_area"
print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
morph = fit_morphology(
    mm.mesh,
    skeleton,
    options=FitOptions(
        max_edge_length=max_edge_length,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=False,
    ),
)
# write swc to file
morph.to_swc_file(swc_filepath)
# validation
validator = Validation(mm, skeleton, morph)
validator.full_validation()

Computing skeleton for radius_strategy=equivalent_area ...

INFO:mascaf.graph_fitting:Tracing done: nodes=15, edges=14, samples=19, section=0, fallback=0 (0.0%)
INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 15 nodes, 14 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 3.5925
INFO:mascaf.validation:---- Ratio:             0.7034
INFO:mascaf.validation:---- Error:             -1.5147
INFO:mascaf.validation:---- Relative error:    -29.66%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   18.7642
INFO:mascaf.validation:---- Ratio:             0.8408
INFO:mascaf.validation:---- Error:             -3.5534


In [7]:
# plot using swctools
from swctools import SWCModel, plot_model

model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
title = f"Branching Skeleton"
fig = plot_model(swc_model=model, title=title, plot_endcaps=True)
fig.show()

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=15 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=15 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=15 edges=14 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=14 sides=16 end_caps=False verts=448 faces=448
INFO:swctools.geometry:FrustaSet.from_swc_model edges=14 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=14 show_frusta=True show_centroid=True


SWCModel: nodes=15, edges=14, components=1, cycles=0, branch_points=2, roots=1, leaves=3, self_loops=0, density=0.1333


In [6]:
help(plot_model)

Help on function plot_model in module swctools.viz:

plot_model(*, swc_model: 'SWCModel | None' = None, frusta: 'FrustaSet | None' = None, show_frusta: 'bool' = True, show_centroid: 'bool' = True, title: 'str | None' = None, sides: 'int' = 16, end_caps: 'bool' = False, plot_endcaps: 'bool' = False, color: 'str' = 'lightblue', opacity: 'float' = 0.8, flatshading: 'bool' = True, tag_colors: 'dict[int, str] | None' = None, radius_scale: 'float' = 1.0, slider: 'bool' = False, min_scale: 'float' = 0.0, max_scale: 'float' = 1.0, steps: 'int' = 21, centroid_color: 'str' = '#1f77b4', centroid_line_width: 'float' = 2.0, show_nodes: 'bool' = False, node_size: 'float' = 2.0, point_set: 'PointSet | None' = None, point_size: 'float' = 1.0, point_color: 'str' = '#d62728', output_path: 'str | None' = None, auto_open: 'bool' = False, width: 'int' = 1200, height: 'int' = 900, hide_axes: 'bool' = False) -> 'go.Figure'
    Master visualization combining centroid, frusta, slider, and overlay points.

    